# ADDING FEATURES IN PRODUCTS TABLE

In [7]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
import os

def create_residual_features(df, col_x='price', col_y='cogs'):
    """
    Tạo 2 cột mới:
    - residual_log: phần dư sau hồi quy tuyến tính trên log1p(price) và log1p(cogs)
    - residual_pct_log: phần dư tương đối (dưới dạng tỷ lệ phần trăm)
    """
    # Log transform
    df_log = df.copy()
    df_log['log_price'] = np.log1p(df[col_x])
    df_log['log_cogs'] = np.log1p(df[col_y])
    
    # Hồi quy tuyến tính
    X = df_log[['log_price']].values
    y = df_log['log_cogs'].values
    model = LinearRegression().fit(X, y)
    y_pred = model.predict(X)
    
    # Residual tuyệt đối trên log scale
    residual_log = y - y_pred
    # Residual tương đối (phần trăm): trở về scale gốc? Thông thường dùng log-scale thì residual_log đã là chênh lệch log,
    # nhưng để dễ hiểu ta tính phần trăm sai số trên giá trị gốc.
    # Cách chính xác: residual_pct = (exp(y) - exp(y_pred))/exp(y_pred) * 100 ≈ 100 * (exp(residual_log)-1)
    # Vì log1p gần với log, nhưng để đơn giản, ta tính trên giá trị gốc:
    cogs_actual = df[col_y]
    cogs_pred = np.expm1(y_pred)  # vì y_pred là log1p(cogs), expm1 để về cogs
    residual_pct = (cogs_actual - cogs_pred) / cogs_pred * 100
    
    df['residual_log_price_cogs'] = residual_log
    df['residual_pct_price_cogs'] = residual_pct
    
    print("Đã tạo hai cột feature engineering:")
    print("  - residual_log_price_cogs: phần dư trên log scale, dùng cho mô hình tuyến tính chuẩn.")
    print("  - residual_pct_price_cogs: sai số phần trăm so với dự báo, giúp so sánh giữa các mức giá.")
    return df

In [8]:
products = pd.read_csv('../../datathon-2026-round-1/master/products.csv')

output_path = '../data_after_adding_features/master/products.csv'

products_adding_features = create_residual_features(df=products)
products_adding_features.to_csv(output_path)

Đã tạo hai cột feature engineering:
  - residual_log_price_cogs: phần dư trên log scale, dùng cho mô hình tuyến tính chuẩn.
  - residual_pct_price_cogs: sai số phần trăm so với dự báo, giúp so sánh giữa các mức giá.
